In [ ]:
from typing import List
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel

load_dotenv()
@function_tool
def send_email(subject: str, body: str):
    print(subject, body)

@function_tool
def printTool(content: str):
    print('IN THE PRINT TOOL')
    print(content)
    
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

class AgentOrchestation:



    def getTools(self) -> List:

        agent1 = Agent(
            name="historyAgent",
            instructions="You create historical sounding city names.",
            model="gpt-4o-mini"
        )

        gemini_key = os.getenv("GOOGLE_API_KEY")

        geminiClient = AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=gemini_key
        )

        geminiModel = OpenAIChatCompletionsModel(
            model="gemini-2.0-flash",
            openai_client=geminiClient
        )

        agent2 = Agent(
            name="funnyAgent",
            instructions="You create funny city names.",
            model=geminiModel
        )

        agent1Tool = agent1.as_tool(
            tool_name="nameAgent1",
            tool_description="Provide a historical city name"
        )

        agent2Tool = agent2.as_tool(
            tool_name="nameAgent2",
            tool_description="Provide a funny city name"
        )

        return [agent1Tool, agent2Tool, send_email]

    async def agentRunner(self):
        agentHandover = Agent(
            name="handoff agent",
            instructions="You give a description for the whatever is given to you and enhance it and then print it using the tool given to you",
            model="gpt-4o-mini",
            tools=[printTool],
            handoff_description="print the details given to you"
        )
        instructions="""
You help people with requests.

If the user asks for a city name, use one of the naming tools.

After generating the name, handoff to the handoff agent to enhance the description and print it.
"""
        agentNew = Agent(
            name="ThirdAgent",
            instructions=instructions,
            model="gpt-4o-mini",
            tools=self.getTools(),
            handoffs=[agentHandover],
            input_guardrails=[guardrail_against_name]
        )

        result = await Runner.run(
            agentNew,
            "Give a name of a Himalayan city located on the side of river Satluj at height of 1500m surrounded by snowclad mountains.Give it for Ambuj"
        )

        return result


orch = AgentOrchestation()
result = await orch.agentRunner()


IN THE PRINT TOOL
### Himalayan City Name Suggestions
1. **Valmira Khasa**  
2. **Satdhar Pur**  
3. **Shivashray**  
4. **Nivalis Heights**  
5. **Brahmavarta**  
6. **Thundering Glade**  
7. **Himalora**  
8. **Kundalitha**  
9. **Suryakot**  
10. **Rajyamala**  

These names reflect the rich cultural and geographical elements of a city located at the height of 1500m beside the river Satluj, surrounded by majestic, snowclad mountains.
